# 亚洲同日信息与美股剩余时段
旧收盘到收盘结果不变。本卡只使用假定已在亚洲当地18:00可得的当前收盘，预测美股09:47之后至当日收盘的未发生收益。相同五市场、Ridge10、6次配对拟合；两组共享可用性与观察间隔控制，不补造亚洲假日数据。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/asia_same_session_information_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.asia_same_session_information import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'asia_same_session_information_20261014',
 'plan_sha256': '7b7ca0f62604a48499bd659ba49b7fce033ef469d72514a105bd2d3e8fa9d99e',
 'decision': 'not_promoted_fixed_asia_remaining_session_failed',
 'information_gate_pass': False,
 'objective_achieved': False,
 'new_fits': 6,
 'new_backtests': 0,
 'rows': 2006,
 'unknown_feature_rows': {'twii': 132,
  'tsmc_tw': 133,
  'skhynix': 112,
  'kospi': 111,
  'nikkei': 131},
 'actual_provider_first_receipt_certified': False,
 'source_timezone_labels_only': True,
 'availability_role': 'conservative vendor daily bar availability assumption, not actual first receipt or vintage certification',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/asia_same_session_information_20261014',
 'price_provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013',
 'region': 'us',
 'as_of': '2026-10-09',
 'recorder_id': '70fcc8953904470892ccaf009e68b53c',
 'experiment_id': '752994775491725388',
 'recorder_uri': '

# 亚洲同日信息：修正时间解释后，固定增量仍未通过

旧研究的美股标签为收盘到收盘，包含亚洲当日收盘之前已发生的收益，所以旧同日相关不能直接交易整个区间；但“所有同日信息均不可交易”的表述过宽。亚洲现金市场收盘早于同日本地日期的美股开盘，是否可预测之后的收益需要另行检验。旧统计、旧计划、Recorder与未晋升决定保留；仅对旧代码说明和结论追加范围更正，不重跑lag1切换路线。

本卡0组合回测、6次固定拟合：2021/2022/2023截止各拟合控制与候选，Ridge10、原五个亚洲市场不作市场选择。预测目标是美股09:47开盘至当日实际收盘的现金/SOXL相对SOXX对数收益，09:46决策。原生Qlib日历提供半日市的实际结束时刻，标签在该时刻+60秒后成熟；折边界按US trade_date解释。因此有效评分252/251/945行，不能与旧H5成熟掩码下的行数或指标直接相减。

## 时间与缺失合同

- 原始文件按交易所本地日期标记。采用当地18:00可用的保守研究假设，逐行转换到纽约时区并检查早于09:46；台湾夏令时对应纽约06:00、冬令时05:00，韩国/日本更早。
- 交易时间参考：[TWSE](https://www.twse.com.tw/en/about/company/service.html)、[JPX](https://www.jpx.co.jp/english/equities/trading/domestic/01.html)、[KRX](https://global.krx.co.kr/contents/GLB/01/0109/0109000000/guide_to_trading_in_the_korean_stock_market.pdf)。交易所收市时间不等于Yahoo历史首收时间，18:00是假设，不是供应商认证。
- 变化率按本地序列“本次收盘/上次真实观察收盘-1”计算，避免在美亚联合日历上把一个缺日错误传成次日收益也缺失；没有前值填充假日。本地观察间隔天数与可用性标志在两组中完全相同，候选仅多五个数值变化率。
- 2006个共同市场日全部保留；五市场未知输入行分别132/133/112/111/131，原始未知保留NaN，只有模型内部使用训练均值编码。间隔可能包含休市或供应商缺失，不一概称作单个交易日收益。
- 亚洲原始单一版本数据、供应商首发/修订和具体盘中时间戳未获认证；官方时区证明不能补足该资格。完整源字节、新Qlib特征provider与时点表已保存NAS。

## 结果

标准化MSE差为控制误差减候选误差，正数才是改善，非收益百分点；使用相同控制训练标签尺度。

|窗口|观察数|联合误差改善|95%区块区间|
|---|---:|---:|---:|
|development_2021|252|0.015293|[-0.013207, 0.050104]|
|development_2022|251|0.029297|[-0.005940, 0.062760]|
|reused_outer|945|-0.051304|[-0.093190, -0.013844]|
|pooled_development|503|0.022281|[-0.000393, 0.045977]|

开发期两年点估计均改善，但合并下界-0.000393仍未满足预定大于0门；不会改用90%区间或放松数值门槛。2023年后联合误差恶化约0.0513标准化单位，区块区间也为负。样本外方向不稳定，不能据开发期改善选择该候选。区块平均长63会话、2000次，未校正全部历史搜索。

决定：**not_promoted_fixed_asia_remaining_session_failed**。这次否定的是固定五市场/Ridge10/剩余时段增量，不是把新时间解释重新宣称成前视。无需为失败信息再跑组合账户，不改期限、市场、符号、alpha或可用时刻继续搜索。稳定净SOXX超额目标未完成，SIM未改。

下一信息需求应具有不同的可验证机制，并先明确输入可用时刻和目标收益尚未发生的区间；已完成的旧lag1路线与本卡当前日剩余时段路线分别保留负面证据。任何未来通过的信息仍须在Tiered完整账户和独立前向中验证，不能由相关或MSE替代。

[预测指标](forecast_metrics.csv)、[信息增量](information_increment.csv)、[训练成熟度](training_audit.csv)、[对象清单](verification_manifest.json)。DST、本地观测间隔与缺失合同测试通过；Notebook、原生特征读回及NAS校验通过。旧卡只做范围说明更新，不重新拟合或回放。
